# Data Cleaning

In [11]:
df_info = pd.read_csv(
    os.path.join(DATA_DIR, sample_info),
    header=None,
    names=["type", "key", "value"],
    engine="python",        
    on_bad_lines='warn'     
)

print(df_info)

       type              key                                      value
0   version            2.1.0                                       None
1      info   balls_per_over                                          6
2      info             team                        Sunrisers Hyderabad
3      info             team                Royal Challengers Bangalore
4      info           gender                                       male
5      info           season                                       2017
6      info             date                                 2017/04/05
7      info            event                      Indian Premier League
8      info     match_number                                          1
9      info            venue  Rajiv Gandhi International Stadium, Uppal
10     info             city                                  Hyderabad
11     info      toss_winner                Royal Challengers Bangalore
12     info    toss_decision                                    

C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\967468095.py:1: ParserWarning: Skipping line 22: Expected 3 fields in line 22, saw 4

  df_info = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\967468095.py:1: ParserWarning: Skipping line 23: Expected 3 fields in line 23, saw 4

  df_info = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\967468095.py:1: ParserWarning: Skipping line 24: Expected 3 fields in line 24, saw 4

  df_info = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\967468095.py:1: ParserWarning: Skipping line 25: Expected 3 fields in line 25, saw 4

  df_info = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\967468095.py:1: ParserWarning: Skipping line 26: Expected 3 fields in line 26, saw 4

  df_info = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\967468095.py:1: ParserWarning: Skipping line 27: Expected 3 fields in line 27, saw 4

  df_info =

In [12]:
def parse_info_file(info_path):
    df = pd.read_csv(
        info_path,
        header=None,
        names=["type", "key", "value"],
        engine="python",
        on_bad_lines='warn'
    )
    df = df[df["type"] == "info"]

    info = {}

    # Simple single-value fields
    for field in ["city", "venue", "gender", "match_type",
                  "toss_winner", "toss_decision", "season",
                  "event", "match_number"]:
        row = df[df["key"] == field]
        info[field] = row["value"].values[0] if not row.empty else None

    # Date key is 'date' (not 'dates')
    date_row = df[df["key"] == "date"]
    info["match_date"] = date_row["value"].values[0] if not date_row.empty else None

    # Winner is a direct row
    winner_row = df[df["key"] == "winner"]
    info["winner"] = winner_row["value"].values[0] if not winner_row.empty else None

    # Win margin — only one of these will exist per match
    runs_row = df[df["key"] == "winner_runs"]
    wickets_row = df[df["key"] == "winner_wickets"]
    info["win_by_runs"] = runs_row["value"].values[0] if not runs_row.empty else None
    info["win_by_wickets"] = wickets_row["value"].values[0] if not wickets_row.empty else None

    # Result type — if no winner row, check for no result / tie
    if info["winner"] is None:
        no_result = df[df["key"] == "no_result"]
        tie = df[df["key"] == "tie"]
        if not no_result.empty:
            info["result"] = "no result"
        elif not tie.empty:
            info["result"] = "tie"
        else:
            info["result"] = "unknown"
    else:
        info["result"] = "normal"

    # Player of match — can be multiple, join with comma
    potm = df[df["key"] == "player_of_match"]
    info["player_of_match"] = ", ".join(potm["value"].values.astype(str)) if not potm.empty else None

    # Teams
    teams = df[df["key"] == "team"]["value"].values
    info["team1"] = teams[0] if len(teams) > 0 else None
    info["team2"] = teams[1] if len(teams) > 1 else None

    return info

In [13]:
import glob
from tqdm import tqdm

OUTPUT_PATH = r"C:\Users\A_R COMPUTERS\Desktop\Vineet\Project\IPL Prediction\ipl_master.csv"

def merge_all_matches(data_dir, output_path):
    bbb_files = [
        f for f in glob.glob(os.path.join(data_dir, "*.csv"))
        if "_info" not in f
    ]
    print(f"Found {len(bbb_files)} ball-by-ball files")

    all_dfs = []
    skipped = []

    for bbb_path in tqdm(bbb_files, desc="Merging"):
        match_id = os.path.basename(bbb_path).replace(".csv", "")
        info_path = os.path.join(data_dir, f"{match_id}_info.csv")

        if not os.path.exists(info_path):
            skipped.append(match_id)
            continue

        try:
            bbb_df = pd.read_csv(bbb_path)
            info_dict = parse_info_file(info_path)

            bbb_df["match_id"] = match_id
            for key, val in info_dict.items():
                bbb_df[key] = val

            all_dfs.append(bbb_df)

        except Exception as e:
            print(f"\nError on {match_id}: {e}")
            skipped.append(match_id)

    master_df = pd.concat(all_dfs, ignore_index=True)
    master_df.to_csv(output_path, index=False)

    print(f"\nDone!")
    print(f"  Matches merged : {len(all_dfs)}")
    print(f"  Total rows     : {len(master_df):,}")
    print(f"  Skipped        : {len(skipped)}")
    if skipped:
        print(f"  Skipped IDs    : {skipped[:5]} ...")
    print(f"  Saved to       : {output_path}")

    return master_df

In [ ]:
master = merge_all_matches(DATA_DIR, OUTPUT_PATH)

Found 1175 ball-by-ball files


Merging:   0%|          | 0/1175 [00:00<?, ?it/s]C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\282976163.py:2: ParserWarning: Skipping line 22: Expected 3 fields in line 22, saw 4

  df = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\282976163.py:2: ParserWarning: Skipping line 23: Expected 3 fields in line 23, saw 4

  df = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\282976163.py:2: ParserWarning: Skipping line 24: Expected 3 fields in line 24, saw 4

  df = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\282976163.py:2: ParserWarning: Skipping line 25: Expected 3 fields in line 25, saw 4

  df = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\282976163.py:2: ParserWarning: Skipping line 26: Expected 3 fields in line 26, saw 4

  df = pd.read_csv(
C:\Users\A_R COMPUTERS\AppData\Local\Temp\ipykernel_11396\282976163.py:2: ParserWarning: Skipping line 27: Expected 3 fields in lin

In [15]:
master

,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,event,match_number,match_date,winner,win_by_runs,win_by_wickets,result,player_of_match,team1,team2
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.1,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,...,Indian Premier League,1,2017/04/05,Sunrisers Hyderabad,35,None,normal,Yuvraj Singh,Sunrisers Hyderabad,Royal Challengers Bangalore
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.2,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,...,Indian Premier League,1,2017/04/05,Sunrisers Hyderabad,35,None,normal,Yuvraj Singh,Sunrisers Hyderabad,Royal Challengers Bangalore
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.3,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,...,Indian Premier League,1,2017/04/05,Sunrisers Hyderabad,35,None,normal,Yuvraj Singh,Sunrisers Hyderabad,Royal Challengers Bangalore
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.4,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,...,Indian Premier League,1,2017/04/05,Sunrisers Hyderabad,35,None,normal,Yuvraj Singh,Sunrisers Hyderabad,Royal Challengers Bangalore
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.5,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,...,Indian Premier League,1,2017/04/05,Sunrisers Hyderabad,35,None,normal,Yuvraj Singh,Sunrisers Hyderabad,Royal Challengers Bangalore
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279581,981019,2016,2016-05-29,M Chinnaswamy Stadium,2,19.2,Royal Challengers Bangalore,Sunrisers Hyderabad,Sachin Baby,CJ Jordan,...,Indian Premier League,None,2016/05/29,Sunrisers Hyderabad,8,None,normal,BCJ Cutting,Royal Challengers Bangalore,Sunrisers Hyderabad
279582,981019,2016,2016-05-29,M Chinnaswamy Stadium,2,19.3,Royal Challengers Bangalore,Sunrisers Hyderabad,Sachin Baby,CJ Jordan,...,Indian Premier League,None,2016/05/29,Sunrisers Hyderabad,8,None,normal,BCJ Cutting,Royal Challengers Bangalore,Sunrisers Hyderabad
279583,981019,2016,2016-05-29,M Chinnaswamy Stadium,2,19.4,Royal Challengers Bangalore,Sunrisers Hyderabad,Iqbal Abdulla,Sachin Baby,...,Indian Premier League,None,2016/05/29,Sunrisers Hyderabad,8,None,normal,BCJ Cutting,Royal Challengers Bangalore,Sunrisers Hyderabad
279584,981019,2016,2016-05-29,M Chinnaswamy Stadium,2,19.5,Royal Challengers Bangalore,Sunrisers Hyderabad,Sachin Baby,Iqbal Abdulla,...,Indian Premier League,None,2016/05/29,Sunrisers Hyderabad,8,None,normal,BCJ Cutting,Royal Challengers Bangalore,Sunrisers Hyderabad


In [16]:
print(master[["match_id", "season", "team1", "team2", "winner",
              "toss_winner", "toss_decision", "venue",
              "win_by_runs", "win_by_wickets"]].drop_duplicates("match_id").head(10))

     match_id season                        team1  \
0     1082591   2017          Sunrisers Hyderabad   
248   1082592   2017       Rising Pune Supergiant   
495   1082593   2017                Gujarat Lions   
713   1082594   2017              Kings XI Punjab   
960   1082595   2017  Royal Challengers Bangalore   
1208  1082596   2017          Sunrisers Hyderabad   
1424  1082597   2017               Mumbai Indians   
1678  1082598   2017              Kings XI Punjab   
1890  1082599   2017       Rising Pune Supergiant   
2116  1082600   2017               Mumbai Indians   

                            team2                       winner  \
0     Royal Challengers Bangalore          Sunrisers Hyderabad   
248                Mumbai Indians       Rising Pune Supergiant   
495         Kolkata Knight Riders        Kolkata Knight Riders   
713        Rising Pune Supergiant              Kings XI Punjab   
960              Delhi Daredevils  Royal Challengers Bangalore   
1208                

In [17]:
match_summary = master[["match_id", "season", "team1", "team2", "winner",
              "toss_winner", "toss_decision", "venue",
              "win_by_runs", "win_by_wickets"]].drop_duplicates("match_id")

print("Total unique matches:", len(match_summary))
print("Seasons covered:", sorted(master["season"].dropna().unique()))
print("\nSample:\n")
print(match_summary.head(10).to_string())

Total unique matches: 1175
Seasons covered: ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Sample:

     match_id season                        team1                        team2                       winner                  toss_winner toss_decision                                      venue win_by_runs win_by_wickets
0     1082591   2017          Sunrisers Hyderabad  Royal Challengers Bangalore          Sunrisers Hyderabad  Royal Challengers Bangalore         field  Rajiv Gandhi International Stadium, Uppal          35           None
248   1082592   2017       Rising Pune Supergiant               Mumbai Indians       Rising Pune Supergiant       Rising Pune Supergiant         field    Maharashtra Cricket Association Stadium        None              7
495   1082593   2017                Gujarat Lions        Kolkata Knight Riders        Kolkata Knight Riders        Kolkat

In [18]:
match_summary = master[["match_id", "season", "team1", "team2", "winner",
              "toss_winner", "toss_decision", "venue",
              "win_by_runs", "win_by_wickets"]].drop_duplicates("match_id")

print("Total unique matches:", len(match_summary))
print("Seasons covered:", sorted(master["season"].dropna().unique()))
print("\nSample:\n")
print(match_summary.head(10).to_string())

Total unique matches: 1175
Seasons covered: ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Sample:

     match_id season                        team1                        team2                       winner                  toss_winner toss_decision                                      venue win_by_runs win_by_wickets
0     1082591   2017          Sunrisers Hyderabad  Royal Challengers Bangalore          Sunrisers Hyderabad  Royal Challengers Bangalore         field  Rajiv Gandhi International Stadium, Uppal          35           None
248   1082592   2017       Rising Pune Supergiant               Mumbai Indians       Rising Pune Supergiant       Rising Pune Supergiant         field    Maharashtra Cricket Association Stadium        None              7
495   1082593   2017                Gujarat Lions        Kolkata Knight Riders        Kolkata Knight Riders        Kolkat

In [19]:
!pip install requests beautifulsoup4

In [22]:
all_teams = set(master["team1"].dropna().unique()) | \
            set(master["team2"].dropna().unique()) | \
            set(master["winner"].dropna().unique())

for t in sorted(all_teams):
    print(t)

Chennai Super Kings
Deccan Chargers
Delhi Capitals
Delhi Daredevils
Gujarat Lions
Gujarat Titans
Kings XI Punjab
Kochi Tuskers Kerala
Kolkata Knight Riders
Lucknow Super Giants
Mumbai Indians
Pune Warriors
Punjab Kings
Rajasthan Royals
Rising Pune Supergiant
Rising Pune Supergiants
Royal Challengers Bangalore
Royal Challengers Bengaluru
Sunrisers Hyderabad


In [23]:
team_name_map = {

    # ── Active teams — name changes ───────────────────────────────
    "Delhi Daredevils"          : "Delhi Capitals",       # renamed 2019
    "Kings XI Punjab"           : "Punjab Kings",         # renamed 2021
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",  # renamed 2023

    # ── Defunct teams — played for a few seasons then shut down ───
    "Deccan Chargers"           : "Sunrisers Hyderabad",  # replaced by SRH in 2013
    "Kochi Tuskers Kerala"      : "Defunct",              # only 2011
    "Pune Warriors"             : "Defunct",              # 2011–2013
    "Rising Pune Supergiant"    : "Defunct",              # 2016–2017
    "Rising Pune Supergiants"   : "Defunct",              # spelling variant
    "Gujarat Lions"             : "Defunct",              # 2016–2017
}

In [24]:
def standardize_team_names(df, mapping):
    for col in ["team1", "team2", "winner", "toss_winner"]:
        if col in df.columns:
            df[col] = df[col].replace(mapping)
    return df

# Apply
master = standardize_team_names(master, team_name_map)

# Verify — print all unique team names after cleaning
all_teams_after = set(master["team1"].dropna().unique()) | \
                  set(master["team2"].dropna().unique())

print("Teams after standardization:")
for t in sorted(all_teams_after):
    print(f"  {t}")

Teams after standardization:
  Chennai Super Kings
  Defunct
  Delhi Capitals
  Gujarat Titans
  Kolkata Knight Riders
  Lucknow Super Giants
  Mumbai Indians
  Punjab Kings
  Rajasthan Royals
  Royal Challengers Bengaluru
  Sunrisers Hyderabad


In [26]:
master.to_csv(MASTER_CSV, index=False)
print("Cleaned master saved!")
print("Total rows      :", len(master))
print("Unique teams    :", master["team1"].nunique())
print("Seasons covered :", sorted(master["season"].dropna().unique()))

Cleaned master saved!
Total rows      : 279586
Unique teams    : 11
Seasons covered : ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']


In [27]:
defunct_matches = master[
    (master["team1"] == "Defunct") | (master["team2"] == "Defunct")
]["match_id"].nunique()

active_matches = master[
    (master["team1"] != "Defunct") & (master["team2"] != "Defunct")
]["match_id"].nunique()

print(f"Matches involving defunct teams : {defunct_matches}")
print(f"Matches between active teams    : {active_matches}")

Matches involving defunct teams : 115
Matches between active teams    : 1060


In [28]:
print("Shape           :", master.shape)
print("Total rows      :", len(master))
print("Unique matches  :", master["match_id"].nunique())
print("Seasons covered :", sorted(master["season"].dropna().unique()))
print("\nTeams after cleaning:")
all_teams = set(master["team1"].dropna().unique()) | set(master["team2"].dropna().unique())
for t in sorted(all_teams):
    print(f"  {t}")

print("\nMissing values in key columns:")
print(master[["winner", "toss_winner", "venue", "season", "team1", "team2"]].isnull().sum())

Shape           : (279586, 37)
Total rows      : 279586
Unique matches  : 1175
Seasons covered : ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Teams after cleaning:
  Chennai Super Kings
  Defunct
  Delhi Capitals
  Gujarat Titans
  Kolkata Knight Riders
  Lucknow Super Giants
  Mumbai Indians
  Punjab Kings
  Rajasthan Royals
  Royal Challengers Bengaluru
  Sunrisers Hyderabad

Missing values in key columns:
winner         4702
toss_winner       0
venue             0
season            0
team1             0
team2             0
dtype: int64
